# `paper_tmlr_1` scale-up pilot — Colab harness

Runs a **5-cell focused pilot** comparing the SPLM family against a parameter-matched all-attention baseline on TinyStories at the existing E9 scale-up configuration:

- **d** = 256, **L** = 8, **max_len** = 1024, **block_size** = 512, **batch_size** = 16
- **steps** = 8000, AdamW(0.9, 0.95), lr = 5e-4 cosine, 400-step warmup
- TinyStories, ~5 M GPT-2 BPE training tokens, ~140 k validation tokens
- mass_mode = `logfreq`, alpha-init 0.1, surprisal precomputed on TinyStories
- Single seed (seed 0) per arm in this pilot pass

## Arms

| # | Arm | Trainer | Notes |
|---|-----|---------|-------|
| 1 | matched-attn baseline    | `train_matched_baseline_scaleup.py`  | All-attention GPT-2-style decoder (d=256, n_head=4, mlp_mult=4, ~19.5 M params) |
| 2 | SPLM em_ln (all-SPLM, TF32 off) | `train_splm_em_ln_scaleup.py`  | Existing E9 SPLM arm (γ free or fixed γ=0.30); TF32 disabled by default for autograd.grad numerical stability |
| 2b | SPLM em_ln (TF32 on, artifact ref) | `train_splm_em_ln_scaleup.py --allow-tf32` | Optional ~12 min H100 cell that produces the precision-artifact reference so the aggregator can empirically test "TF32 inflates PPL" |
| 3 | Helmholtz Q9d AAAASSSS   | `train_helmholtz_scaleup.py`         | H1 winner at small scale |
| 4 | Hybrid Variant A (k=4, m=4) | `train_hybrid_scaleup.py`         | H1 winner at small scale |
| 5 | PARF Q9c sparse k=4      | `train_parf_scaleup.py`              | P5 winner at small scale (Gumbel-softmax sparse routing); V_φ shrunk to H=16 to fit A100 |
| 5b | PARF Q9c sparse k=4 (full V_φ) | `train_parf_scaleup.py`        | Optional H100 80GB rerun at H=128 for the paper_tmlr_1 discussion section |

## Decision rule

Δ = PPL(matched-attn) − PPL(arm). With **Δ_min = 5 PPL**:
- Δ > +5 ⇒ SPLM-family arm beats attention baseline at scale
- |Δ| ≤ 5 ⇒ tie
- Δ < −5 ⇒ baseline wins for that arm

## Wall-clock estimates (A100 40 GB; multiply by ~1.7× on L4)

| Arm | Per-cell wall-clock |
|---|---:|
| matched-attn baseline    | ~2.5 h |
| SPLM em_ln (Arm 2, TF32 off) | ~3.5 h |
| SPLM em_ln (Arm 2b, TF32 on, H100 only) | ~12 min (skip on A100) |
| Helmholtz Q9d            | ~3.5 h |
| Hybrid VA                | ~3.0 h |
| PARF Q9c sparse k=4      | ~50-85 min (A100, H=16, grad-accum=2) |
| PARF Q9c sparse k=4 (5b) | ~60-90 min (H100 80GB+, H=128, grad-accum=2) |
| **Total (Arms 1-5, A100)** | **~13-15 h** |

On Colab Pro+ with A100, a single ~24 h session is enough to complete the full pilot. If a session disconnects, the notebook is **idempotent**: it skips arms whose `*_summary.md` already exists in Drive.

## 1. Environment setup

Mount Drive (where checkpoints + logs persist), clone the repo into Colab's ephemeral disk, and install Python deps.

In [ ]:
import os, sys, subprocess, shutil, json, time
from pathlib import Path

# Set the CUDA allocator config BEFORE any torch import or CUDA context
# creation.  expandable_segments=True helps the caching allocator grow
# and shrink segments to reduce fragmentation.  This is propagated to
# every trainer subprocess we launch via subprocess.Popen, which
# inherits the parent process's environment.
os.environ.setdefault('PYTORCH_ALLOC_CONF', 'expandable_segments:True')
print('PYTORCH_ALLOC_CONF =', os.environ['PYTORCH_ALLOC_CONF'])

IN_COLAB = 'google.colab' in sys.modules
print('In Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/semsimula_pilot')
    REPO_PARENT  = Path('/content')
else:
    DRIVE_ROOT   = Path.home() / 'semsimula_pilot'
    REPO_PARENT  = Path.cwd().parent.parent.parent.parent  # local fallback

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
DRIVE_LOGS    = DRIVE_ROOT / 'logs'
DRIVE_LOGS.mkdir(parents=True, exist_ok=True)
print('Drive root   :', DRIVE_ROOT)
print('Results dir  :', DRIVE_RESULTS)
print('Logs dir     :', DRIVE_LOGS)

In [ ]:
# Clone (or pull) the semsimula repo into Colab's ephemeral disk.
# Replace REPO_URL with your fork or branch as needed.
REPO_URL  = 'https://github.com/dimitarpg13/semsimula.git'
REPO_NAME = 'semsimula'
REPO_DIR  = REPO_PARENT / REPO_NAME

if not REPO_DIR.exists():
    print(f'Cloning {REPO_URL} -> {REPO_DIR} ...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f'{REPO_DIR} already exists; pulling latest ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=False)

SCALEUP_DIR = REPO_DIR / 'notebooks' / 'conservative_arch' / 'scaleup'
assert SCALEUP_DIR.exists(), f'scaleup dir not found at {SCALEUP_DIR}'
print('scaleup dir  :', SCALEUP_DIR)
print('contents     :', sorted(p.name for p in SCALEUP_DIR.iterdir())[:25])

In [ ]:
# Install Python dependencies. Colab already ships with a recent torch+CUDA;
# we only need to top up the smaller helpers.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers', 'datasets', 'pyarrow'], check=True)
print('Dependencies OK')

In [ ]:
# Verify GPU + report device specs.
import torch
print('torch      :', torch.__version__)
print('cuda avail :', torch.cuda.is_available())
if torch.cuda.is_available():
    dev = torch.cuda.get_device_properties(0)
    print(f'GPU         : {dev.name}')
    print(f'Total VRAM  : {dev.total_memory / 1e9:.1f} GB')
    print(f'CUDA cap    : sm_{dev.major}{dev.minor}')
    print(f'CUDA driver : {torch.version.cuda}')
else:
    print('NO CUDA GPU — switch to a GPU runtime: Runtime > Change runtime type > GPU (A100 recommended)')
    raise SystemExit(1)

In [ ]:
# Verify data + logfreq surprisal files are present (tracked in repo).
DATA_DIR    = REPO_DIR / 'notebooks' / 'conservative_arch' / 'data'
LOGFREQ_PATH = SCALEUP_DIR / 'results' / 'logfreq_surprisal_tinystories.npy'
NPZ_PATH    = DATA_DIR / 'tinystories_gpt2_1files_5000000toks.npz'

assert NPZ_PATH.exists(),    f'TinyStories .npz missing at {NPZ_PATH}'
assert LOGFREQ_PATH.exists(), f'logfreq .npy missing at {LOGFREQ_PATH}'
print('TinyStories .npz   :', NPZ_PATH, f'({NPZ_PATH.stat().st_size/1e6:.1f} MB)')
print('logfreq surprisal  :', LOGFREQ_PATH, f'({LOGFREQ_PATH.stat().st_size/1e3:.1f} KB)')

## 2. Run-arm helper

Each arm is launched as a subprocess writing logs and artifacts to `DRIVE_RESULTS`. The helper:

1. **Skips** the run if `<tag>_summary.md` already exists in `DRIVE_RESULTS` (idempotent across session disconnects).
2. **Streams** stdout/stderr to both the notebook output and a Drive log file (so we can audit what happened even if Colab tears down the session).
3. **Writes** all artifacts directly to Drive (not to ephemeral Colab disk).

In [ ]:
import datetime as _dt

def _summary_exists_for(prefix: str, suffix: str = '') -> Path | None:
    """Look for any *_summary.md whose name starts with `prefix` and
    contains `suffix` as a substring. Returns the matching path or None.
    Uses a single-asterisk glob (Python 3.12 rejects adjacent '**' unless
    it is an entire path component) and filters the suffix in Python."""
    for p in sorted(DRIVE_RESULTS.glob(f'{prefix}*_summary.md')):
        if suffix and suffix not in p.name:
            continue
        return p
    return None

def run_arm(name: str, script: str, args: list[str], skip_prefix: str,
            seed: int = 0, log_label: str | None = None,
            extra_skip_suffix: str = '') -> int:
    """Launch a single training arm via subprocess.

    Args:
      name              : human-readable name for logging
      script            : trainer script filename inside SCALEUP_DIR
      args              : list of extra CLI args to pass
      skip_prefix       : prefix used to detect a completed run; if any
                          file in DRIVE_RESULTS matches
                          `{skip_prefix}*_summary.md` we skip.
      seed              : seed value (added to args automatically)
      log_label         : optional label for the Drive log file; defaults
                          to `name` lowercased.
      extra_skip_suffix : additional substring required in the matched
                          summary filename (for arms that share a prefix).
    """
    label = log_label or name.lower().replace(' ', '_').replace('/', '_')
    existing = _summary_exists_for(skip_prefix, extra_skip_suffix)
    if existing is not None:
        print(f'[run] {name}: SKIP (found {existing.name})')
        return 0
    cmd = [sys.executable, '-u', str(SCALEUP_DIR / script),
           '--mode', 'scaleup',
           '--seed', str(seed),
           '--results-dir', str(DRIVE_RESULTS),
           '--tag-suffix', f'seed{seed}',
           ] + list(args)
    ts = _dt.datetime.now().strftime('%Y%m%d_%H%M%S')
    log_path = DRIVE_LOGS / f'{label}_seed{seed}_{ts}.log'
    print(f'[run] {name}')
    print(f'      cmd: {" ".join(cmd)}')
    print(f'      log: {log_path}')
    t0 = time.time()
    with log_path.open('w') as logf:
        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT, text=True,
                                bufsize=1)
        for line in proc.stdout:
            print(line, end='')
            logf.write(line)
            logf.flush()
        rc = proc.wait()
    dt = (time.time() - t0) / 3600.0
    print(f'[run] {name}: exit={rc}  elapsed={dt:.2f} h')
    return rc


## 3. Pre-flight smoke test (5–10 min)

Runs all five trainers in `--mode smoke` (300 steps each, batch=8, block=256). This validates **the entire pipeline** — model build, data load, GPU availability, gradient flow, checkpoint write — at the scale-up parameter count without burning a Colab session. **Do this first** before launching the full pilot.

In [ ]:
SMOKE_RESULTS = DRIVE_ROOT / 'smoke_results'
SMOKE_RESULTS.mkdir(parents=True, exist_ok=True)

def smoke(name: str, script: str, args: list[str]) -> int:
    """Run a trainer in --mode smoke, streaming stdout+stderr to the cell."""
    cmd = [sys.executable, '-u', str(SCALEUP_DIR / script),
           '--mode', 'smoke',
           '--seed', '0',
           '--results-dir', str(SMOKE_RESULTS),
           '--tag-suffix', 'smoke',
           ] + list(args)
    print()
    print(f'[smoke] {name}')
    print(f'      cmd: {" ".join(cmd)}')
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True,
                            bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    rc = proc.wait()
    print(f'[smoke] {name}: exit={rc}')
    return rc

rcs = []
rcs.append(smoke('matched-attn baseline',     'train_matched_baseline_scaleup.py', []))
rcs.append(smoke('SPLM em_ln (all-SPLM)',     'train_splm_em_ln_scaleup.py',       []))
rcs.append(smoke('Helmholtz Q9d AAAASSSS',    'train_helmholtz_scaleup.py',        ['--schedule', 'AAAASSSS']))
rcs.append(smoke('Hybrid VA k=4 m=4',         'train_hybrid_scaleup.py',           ['--n-attn', '4', '--n-splm', '4']))
rcs.append(smoke('PARF Q9c sparse k=4',       'train_parf_scaleup.py',             ['--top-k', '4']))
print()
print('[smoke] return codes:', rcs)
if any(rc != 0 for rc in rcs):
    raise RuntimeError('one or more smoke runs failed; inspect the output above')
print()
print('[smoke] all five arms passed; proceed to full pilot below.')


## 4. Full pilot — five sequential arms

Run each arm cell-by-cell so you can monitor progress and (if needed) interrupt. Each cell is **safe to re-run**: it skips if the corresponding `*_summary.md` already exists in `DRIVE_RESULTS`.

**Order of arms** is intentional:
1. Matched-attention baseline first — locks the reference PPL we will compare against.
2. SPLM em_ln second — replicates the E9 result (this is the existing "globally not" reference).
3. Helmholtz Q9d third — H1 small-scale winner.
4. Hybrid VA fourth — H1 small-scale winner (companion).
5. PARF Q9c sparse last — most expensive cell; if Colab disconnects mid-run we restart this one only.


In [ ]:
# Arm 1 — matched-attention baseline (~2.5 h on A100)
rc = run_arm(
    name='matched-attn baseline',
    script='train_matched_baseline_scaleup.py',
    args=[],
    skip_prefix='matched_baseline_scaleup_scaleup',
    seed=0,
)
assert rc == 0, f'matched-attn baseline returned exit {rc}'

In [ ]:
# Arm 2 — SPLM em_ln (all-SPLM) (~3.5 h on A100)
# Defaults to the E5 winner fixed γ=0.30; pass --fixed-gamma None to use free γ instead.
rc = run_arm(
    name='SPLM em_ln',
    script='train_splm_em_ln_scaleup.py',
    args=['--fixed-gamma', '0.30'],
    skip_prefix='splm_em_ln_scaleup_scaleup',
    seed=0,
)
assert rc == 0, f'SPLM em_ln returned exit {rc}'

In [ ]:
# Arm 2b - SPLM em_ln (TF32 on) precision-artifact reference (~12 min on H100)
#
# OPTIONAL.  Complements Arm 2 (TF32 off, default) with a TF32-on
# rerun so the aggregator can empirically test the "TF32 inflates PPL"
# hypothesis instead of asserting it.  Without this row the TF32
# precision section in PILOT_RESULTS.md cannot draw a conclusion.
#
# History note (May 2026): the four SPLM-family scaleup trainers all
# disable TF32 by default to ensure a clean numerical reference for the
# `torch.autograd.grad(create_graph=True)` second-order path.  This
# cell uses the new --allow-tf32 flag (added in the same commit) to
# leave TF32 enabled, producing the precision-artifact reference.
#
# Tag pattern: splm_em_ln_scaleup_scaleup_tf32on_seed0_*  (different
# from Arm 2's splm_em_ln_scaleup_scaleup_seed0_* so both rows coexist
# in DRIVE_RESULTS and the aggregator picks up both).
#
# Wall-clock on H100: ~12 min (TF32-on is ~2x faster than TF32-off
# matmul, so this is faster than Arm 2).  Skip-if-done is keyed to
# the tf32on_seed0 tag, so the cell is idempotent across reruns.
# NOTE: the trailing `--tag-suffix tf32on_seed0` overrides the
# auto-added `--tag-suffix seed0` from run_arm (argparse takes the last
# occurrence of a duplicated flag).  This produces filenames of the
# form splm_em_ln_scaleup_scaleup_tf32on_seed0_* so they coexist with
# Arm 2's splm_em_ln_scaleup_scaleup_seed0_* artifacts.
rc = run_arm(
    name='SPLM em_ln (TF32 on, artifact reference)',
    script='train_splm_em_ln_scaleup.py',
    args=['--fixed-gamma', '0.30', '--allow-tf32',
          '--tag-suffix', 'tf32on_seed0'],
    skip_prefix='splm_em_ln_scaleup_scaleup_tf32on',
    seed=0,
    log_label='splm_em_ln_tf32on',
)
assert rc == 0, f'SPLM em_ln (TF32 on) returned exit {rc}'


In [ ]:
# Arm 3 — Helmholtz Q9d AAAASSSS (~3.5 h on A100)
rc = run_arm(
    name='Helmholtz Q9d AAAASSSS',
    script='train_helmholtz_scaleup.py',
    args=['--schedule', 'AAAASSSS'],
    skip_prefix='helmholtz_AAAASSSS_L8_scaleup_scaleup',
    seed=0,
)
assert rc == 0, f'Helmholtz Q9d returned exit {rc}'

In [ ]:
# Arm 4 — Hybrid Variant A (k=4, m=4) (~3.0 h on A100)
rc = run_arm(
    name='Hybrid VA k=4 m=4',
    script='train_hybrid_scaleup.py',
    args=['--n-attn', '4', '--n-splm', '4'],
    skip_prefix='hybrid_VA_k4_m4_scaleup_scaleup',
    seed=0,
)
assert rc == 0, f'Hybrid VA returned exit {rc}'

In [ ]:
# Arm 5 — PARF Q9c sparse k=4 (~45-75 min on A100, with --grad-accum 2).
#
# MEMORY NOTE.  The structural V_phi at scaleup has THREE memory
# pressures, each addressed by a different knob:
#
#   (1) V_phi forward state.  Two (B, T, T, H) intermediates per layer.
#       At H=128 these are 2.0 GiB each -> OOM in forward.
#       Fix: scaleup defaults H=16 (256 MiB per intermediate).
#
#   (2) Inner-grad backward.  torch.autograd.grad(U, h_in,
#       create_graph=True) inside _layer_step retains a second-order
#       graph that ~doubles the V_phi forward state.  At H=32 the
#       forward fit but the inner backward came up ~350 MiB short.
#       Fix: covered by the same H=16 reduction.
#
#   (3) Outer loss.backward.  CE loss has gradient (B, T, V) =
#       16 * 512 * 50257 * 4 bytes = 1.54 GiB at scaleup vocab.
#       The full computation graph (all 8 layers' forward + 8 inner
#       second-order graphs + lm_head + CE) plus this 1.54 GiB grad
#       buffer was ~350 MiB over the A100's 40 GiB cliff.
#       Fix: --grad-accum 2 splits the B=16 batch into 2 micro-batches
#       of B=8.  Optim sees the same effective batch (16); per-micro-
#       batch peak memory is halved.  Wall-clock cost: ~1.7x (two
#       independent forward+backward passes per optim step).
#
# Memory at scaleup with H=16 + grad-accum=2 (B_micro=8):
#   per-layer V_phi forward         : ~256 MiB
#   8 layers V_phi working set      : ~6 GiB
#   logits gradient                 : ~768 MiB
#   total per-micro-batch peak      : ~18 GiB
#   headroom on 40 GiB A100         : ~22 GiB  (was ~0 GiB)
#
# If this STILL OOMs (very unlikely), bump --grad-accum to 4 or
# enlarge the V_phi shrink (--v-phi-phi-hidden 8 etc.).
rc = run_arm(
    name='PARF Q9c sparse k=4',
    script='train_parf_scaleup.py',
    args=['--top-k', '4', '--v-phi-kind', 'structural',
          '--grad-accum', '2'],
    skip_prefix='parf_structural',
    seed=0,
    extra_skip_suffix='vphi16_sparse_k4_scaleup_scaleup',
)
assert rc == 0, f'PARF Q9c sparse k=4 returned exit {rc}'

In [ ]:
# Arm 5b — PARF Q9c sparse k=4 at FULL V_phi capacity (H=128).
# *** REQUIRES H100 80 GB or larger.  Will OOM on A100 40 GB.
# *** See Arm 5 above for the reduced-capacity (H=16) A100 variant.
#
# This cell is OPTIONAL and complements Arm 5.  It runs PARF at the
# original validated V_phi widths (phi/theta_hidden=128, mlp_hidden=256)
# with grad-accum=2 (B_micro=8, effective batch=16).
#
# MEMORY MATH (corrected after the first H100 attempt OOMed at 94 GiB).
# The structural V_phi forward retains FOUR (B, T, T, H) tensors per
# layer, not two:
#   1. phi_c_net Linear(1, 128) output    : (B, T, T, 128)
#   2. phi_c_net GELU output              : (B, T, T, 128)
#   3. theta_w broadcast-add hidden       : (B, T, T, 128)
#   4. theta_w GELU output                : (B, T, T, 128)
# Plus autograd.grad(create_graph=True) builds a second graph that
# effectively duplicates the gradient-state references.
#
# Memory budget at H=128, B=16, T=512 (single pass, no grad-accum):
#   per (B, T, T, 128) tensor          : 2.0 GiB
#   4 retained per layer x 8 layers    : 64 GiB   (forward state)
#   inner-grad second-order graph      : +32 GiB  (create_graph=True)
#   model + optimizer + V_theta + ...  : +10 GiB
#   total per-step peak                : ~106 GiB <- OOMs even 96 GB H100
#
# Memory budget at H=128, B=16, T=512 with --grad-accum 2 (B_micro=8):
#   per (B, T, T, 128) tensor          : 1.0 GiB
#   4 retained per layer x 8 layers    : 32 GiB
#   inner-grad second-order graph      : +16 GiB
#   model + optimizer + V_theta + ...  : +10 GiB
#   total per-micro-batch peak         : ~58 GiB  <- fits on 80 GB H100
#
# If you have an H100 with only 40 GB (rare), bump --grad-accum to 4.
# If the run still OOMs on your hardware, drop --v-phi-phi-hidden /
# --v-phi-theta-hidden to 96 (-30% per intermediate, ~25% total).
#
# Wall-clock on H100 with grad-accum=2: ~60-90 min (single optim step
# is two forward+backward passes; H100 is ~2-3x faster than A100 at
# FP32 so per-pass time is comparable to A100 at grad-accum=1).
#
# Different tag from Arm 5 (`vphi128` vs `vphi16`) so both runs coexist
# in DRIVE_RESULTS and the aggregator picks up both rows.  Skip-if-done
# is keyed to `vphi128_sparse_k4_scaleup_scaleup`.
rc = run_arm(
    name='PARF Q9c sparse k=4 (vphi128 H100)',
    script='train_parf_scaleup.py',
    args=['--top-k', '4', '--v-phi-kind', 'structural',
          '--v-phi-phi-hidden', '128',
          '--v-phi-theta-hidden', '128',
          '--v-phi-mlp-hidden', '256',
          '--grad-accum', '2'],
    skip_prefix='parf_structural',
    seed=0,
    log_label='parf_q9c_sparse_k4_vphi128_h100',
    extra_skip_suffix='vphi128_sparse_k4_scaleup_scaleup',
)
assert rc == 0, f'PARF Q9c sparse k=4 (vphi128) returned exit {rc}'

## 5. Comparative analysis

Aggregate the five completed arms into:
- `PILOT_RESULTS.md`  — comparative table + decision-rule annotations
- `pilot_loss_curves.png` — val PPL across arms vs. step
- `pilot_pareto.png`     — params vs. final val PPL

All artifacts go into `DRIVE_RESULTS` so they survive session teardown.

In [ ]:
subprocess.run([sys.executable, str(SCALEUP_DIR / 'aggregate_pilot_results.py'),
                '--results-dir', str(DRIVE_RESULTS),
                '--seed', '0',
                '--out-dir', str(DRIVE_RESULTS)],
               check=True)
print('\n--- PILOT_RESULTS.md ---\n')
print((DRIVE_RESULTS / 'PILOT_RESULTS.md').read_text())

In [ ]:
from IPython.display import Image, display
for img in ('pilot_loss_curves.png', 'pilot_pareto.png'):
    p = DRIVE_RESULTS / img
    if p.exists():
        print(p)
        display(Image(str(p)))
    else:
        print(f'{img} missing — re-run aggregator')

## 6. Persistence & next steps

All artifacts now live at:

```
/content/drive/MyDrive/semsimula_pilot/
├── results/
│   ├── matched_baseline_scaleup_scaleup_seed0_*
│   ├── splm_em_ln_scaleup_scaleup_seed0_*           # Arm 2  (TF32 off, default)
│   ├── splm_em_ln_scaleup_scaleup_tf32on_seed0_*    # Arm 2b (TF32 on, optional artifact ref)
│   ├── helmholtz_AAAASSSS_L8_scaleup_scaleup_seed0_*
│   ├── hybrid_VA_k4_m4_scaleup_scaleup_seed0_*
│   ├── parf_structural_vphi128_sparse_k4_scaleup_scaleup_seed0_*
│   ├── PILOT_RESULTS.md
│   ├── pilot_loss_curves.png
│   └── pilot_pareto.png
└── logs/
    └── <arm>_seed0_<timestamp>.log
```

### What to do with the results

1. **Download `PILOT_RESULTS.md`, `pilot_loss_curves.png`, `pilot_pareto.png`** from Drive to your laptop.
2. **Place them under** `notebooks/conservative_arch/scaleup/results/pilot/` in the local repo.
3. **Commit** the markdown + plots (skip `.pt` checkpoints — they're large and not needed for the paper).
4. **Update `paper_tmlr_1`** Discussion §10 with a one-paragraph summary citing the headline Δ vs the matched baseline.
5. If outcome is **Δ > +5 PPL** for any SPLM-family arm, that becomes a paper-strengthening result.

### If you want a second seed

Re-run the dispatch cells with `seed=1` (the helper's skip-if-done logic ignores other seeds since each seed has its own `_summary.md` filename).